# El objetivo de este notebook

Agruparemos reservas por fecha de estadía (no solo `arrival_date`, sino cada noche ocupada dentro del rango de estancia), contaremos habitaciones ocupadas por día y por hotel, y tomaremos el día de mayor ocupación como proxy de "capacidad total" de ese hotel.

---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os
from src.load import load_to_sqlite

In [ ]:
# Extraemos nuestro historico de reservas de la tabla 'clean_bookings'en la base de datos '\data\hotel_data.db'

def extract_from_sqlite(db_path: str = os.path.join("data", "hotel_data.db"))  -> pd.DataFrame:
    """
    Conecta a la base de datos SQLite y extrae los datos de la tabla 'clean_bookings'.

    Args:
        db_path (str): Ruta al archivo de la base de datos SQLite.

    Returns:
        pd.DataFrame: DataFrame que contiene los datos extraídos.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"❌ Error: No se encontró el archivo de base de datos en {db_path}. Asegúrate de que la base de datos exista.")
    
    print(f"⏳ Conectando a la base de datos en {db_path}...")

    # Crear la conexión a la base de datos SQLite
    engine = create_engine(f'sqlite:///{db_path}')

    # Consulta SQL para extraer los datos de la tabla 'clean_bookings'
    query = "SELECT * FROM clean_bookings"

    print("⏳ Ejecutando la consulta SQL para extraer los datos...")

    # Leer los datos de la tabla especificada en un DataFrame
    df = pd.read_sql_query(query, con=engine)

    print(f"✅ Extracción exitosa. Filas extraídas: {df.shape[0]}, Columnas: {df.shape[1]}")
    
    return df

In [ ]:
df = extract_from_sqlite()

## 1. Expansión a nivel noche-reserva

Nuestro dataset tiene una fila por reserva, no por noche ocupada. Una reserva con `arrival_date` = 5 julio y `total_nights` = 4 ocupa una habitación el 5, 6, 7 y 8 de julio. Si agrupamos por `arrival_date` tal cual, solo contamos cuántas reservas llegaron ese día, no cuántas habitaciones estaban físicamente ocupadas ese día (que incluye llegadas de días anteriores que siguen hospedadas). Para simular capacidad necesitamos lo segundo.

Usaremos columnas que ya creamos en `transform.py`:

In [ ]:
def expand_to_nightly_occupancy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expande cada reserva (no cancelada) en una fila por cada noche ocupada.
    Necesario para calcular ocupación real diaria y simular capacidad de inventario.
    """
    df = df[df['is_canceled'] == 0].copy()  # Descartamos las reservas canceladas

    # Reconstruimos la fecha real de llegada
    df['arrival_date'] = pd.to_datetime(
        df['arrival_date_year'].astype(str) + '-' +
        df['arrival_date_month'].astype(str) + '-' +
        df['arrival_date_day_of_month'].astype(str),
        format='%Y-%B-%d'
    )

    rows = []
    for _, row in df.iterrows():
        for n in range(row['total_nights']):
            rows.append({
                'hotel': row['hotel'],
                'room_type': row['assigned_room_type'],  # Usamos como referencia la habitación que REALMENTE se ocupa, no la que se reserva
                'stay_date': row['arrival_date'] + pd.Timedelta(days=n)
            })

    return pd.DataFrame(rows)

In [ ]:
df_nightly = expand_to_nightly_occupancy(df)

In [ ]:
display(df_nightly.head())

In [ ]:
# Obtenemos la ocupación diaria por hotel
ocupación = df_nightly.groupby('hotel')['stay_date'].count()

In [ ]:
display(ocupacion)

## 2. Calculando el máximo de ocupacion por hotel y tipología

In [ ]:
def calculate_simulated_capacity(df_nightly: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula la capacidad simulada por hotel + tipo de habitación,
    usando el máximo histórico de ocupación diaria observada.
    Incluye sanity check: cuántas veces se alcanzó ese máximo.
    """
    ocupacion_diaria = (
        df_nightly.groupby(['hotel', 'room_type', 'stay_date'])
        .size()
        .reset_index(name='rooms_occupied')
    )

    capacidad = (
        ocupacion_diaria.groupby(['hotel', 'room_type'])['rooms_occupied']
        .max()
        .reset_index(name='capacidad_simulada')
    )

    return capacidad

In [ ]:
inventario = calculate_simulated_capacity(df_nightly)

In [ ]:
display(inventario)

In [ ]:
# Ejecutamos la carga para el inventario

load_to_sqlite(inventario, "inventory", "replace")

In [ ]:
# Ejecutamos la carga para la ocupación diaria
load_to_sqlite(df_nightly, "occupancy", "replace")